# Volatility modelling and cross-market causality

GARCH(1,1) on Organon's returns, an ARIMA fit on the arbitrage spread, and a Granger causality test between Organon and Sun Pharma that respects the two markets' non-overlapping trading hours rather than pairing same-calendar-date returns naively.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg, data, risk
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)

prices = data.load_prices()

def own_returns(ticker):
    c = prices[ticker]["Close"]
    return np.log(c / c.shift(1)).dropna()

ogn_ret = own_returns(cfg.TARGET)
sun_ret = own_returns(cfg.ACQUIRER)


## GARCH(1,1) on Organon returns

Fit separately on the pre- and post-announcement samples rather than across the full history in one pass - a single model spanning the 26-Apr break would average together two different volatility regimes (baseline/run-up vs. deal-pinned) and misrepresent both.

In [2]:

pre_ann = ogn_ret.loc[:"2026-04-24"] * 100   # arch package expects percent-scale returns
post_ann = ogn_ret.loc["2026-04-28":] * 100

garch_pre = risk.fit_garch(pre_ann)
print("GARCH(1,1), pre-announcement sample")
print(garch_pre.summary().tables[1])
print(garch_pre.summary().tables[2])


GARCH(1,1), pre-announcement sample
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0572  6.102e-02     -0.937      0.349 [ -0.177,6.242e-02]
                             Volatility Model                             
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.6236      0.247      2.524  1.161e-02   [  0.139,  1.108]
alpha[1]       0.0953  3.238e-02      2.945  3.234e-03 [3.188e-02,  0.159]
beta[1]        0.8271  5.289e-02     15.638  4.022e-55   [  0.723,  0.931]


In [3]:

try:
    garch_post = risk.fit_garch(post_ann)
    print("GARCH(1,1), post-announcement sample")
    print(garch_post.summary().tables[1])
    print(garch_post.summary().tables[2])
except Exception as e:
    print(f"GARCH did not converge cleanly on the post-announcement sample: {type(e).__name__}: {e}")
    print(f"n_obs post-announcement = {len(post_ann)}; realised vol is already near zero (see market-risk")
    print("notebook, 4.5% annualised) which leaves the model almost nothing to explain - not a bug,")
    print("a direct numerical consequence of the deal-pinning itself.")


GARCH(1,1), post-announcement sample
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu             0.0321  2.490e-02      1.289      0.197 [-1.671e-02,8.091e-02]
                               Volatility Model                              
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega      2.1867e-03  2.797e-03      0.782      0.434 [-3.296e-03,7.669e-03]
alpha[1]   6.9597e-16  6.533e-02  1.065e-14      1.000      [ -0.128,  0.128]
beta[1]        0.9240      0.117      7.885  3.137e-15      [  0.694,  1.154]


In [4]:

persistence_pre = garch_pre.params.get("alpha[1]", np.nan) + garch_pre.params.get("beta[1]", np.nan)
print(f"pre-announcement GARCH persistence (alpha+beta) : {persistence_pre:.4f}")

if persistence_pre < 1:
    # omega/params are on the percent scale the model was fit on (pre_ann = ogn_ret*100);
    # long-run variance here is in percent^2, so it must be converted back to a decimal
    # vol (divide by 100) BEFORE annualising - skipping that step inflates the annualised
    # figure by a factor of 100.
    long_run_daily_var_pct2 = garch_pre.params["omega"] / (1 - persistence_pre)
    long_run_daily_vol_decimal = np.sqrt(long_run_daily_var_pct2) / 100
    long_run_annualised_vol = long_run_daily_vol_decimal * np.sqrt(252)
    print(f"pre-announcement long-run annualised vol implied by the fit : {long_run_annualised_vol:.2%}")
else:
    long_run_annualised_vol = np.nan
    print("  (non-stationary fit - persistence >= 1)")


pre-announcement GARCH persistence (alpha+beta) : 0.9224
pre-announcement long-run annualised vol implied by the fit : 45.00%


## ARIMA on the arbitrage spread

The spread is bounded and mechanically converges toward zero as the deal approaches close (see the market-risk notebook), so this is fit as a description of the post-announcement series' own dynamics, not as a claim that the spread is a stationary, tradeable mean-reverting process in the textbook sense.

In [5]:

from statsmodels.tsa.arima.model import ARIMA

offer = cfg.DEAL["offer_price_usd"]
ogn_close_own = prices[cfg.TARGET]["Close"]
spread = ((offer - ogn_close_own) / ogn_close_own).dropna()
spread_post = spread.loc[spread.index >= pd.Timestamp(cfg.DEAL["announcement_date"])]

arima_fit = ARIMA(spread_post.values, order=(1, 0, 0)).fit()
print(arima_fit.summary().tables[1])
print(f"\nAR(1) coefficient: {arima_fit.params[1]:.4f} - a value below 1 is consistent with the spread")
print("decaying toward its mechanical floor rather than following a random walk, as expected given how it's constructed.")


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0442      0.008      5.318      0.000       0.028       0.060
ar.L1          0.9582      0.046     20.973      0.000       0.869       1.048
sigma2       8.66e-06   1.17e-06      7.408      0.000    6.37e-06     1.1e-05

AR(1) coefficient: 0.9582 - a value below 1 is consistent with the spread
decaying toward its mechanical floor rather than following a random walk, as expected given how it's constructed.


## Granger causality: Organon and Sun Pharma

NSE closes years before NYSE even opens on the same calendar date - roughly 03:45-10:00 UTC for NSE versus 13:30-20:00 UTC for NYSE. That means Sun Pharma's same-date close is genuinely prior information relative to Organon's same-date close, while the reverse is not true: Organon's day-t close (US evening) is only available to the Indian market from day t+1 onward. Two tests follow that respect this ordering, rather than one naive same-day-paired Granger test in each direction.

In [6]:

merged = pd.concat({"ogn": ogn_ret, "sun": sun_ret}, axis=1).dropna()
print(f"shared trading dates : {len(merged)}")


shared trading dates : 1251


### Direction 1: does Organon's past return predict Sun Pharma's return?

Standard Granger causality with lagged Organon values is valid here without adjustment - Organon's close from t-1 (or earlier) is unambiguously known before Sun Pharma's session on day t.

In [7]:

from statsmodels.tsa.stattools import grangercausalitytests

gc_ogn_to_sun = grangercausalitytests(merged[["sun", "ogn"]].values, maxlag=3, verbose=False)
for lag in [1, 2, 3]:
    f_stat, p_val, _, _ = gc_ogn_to_sun[lag][0]["ssr_ftest"]
    print(f"lag {lag}: F={f_stat:.3f}  p={p_val:.4f}  {'significant at 5%' if p_val < 0.05 else 'not significant'}")


lag 1: F=15.274  p=0.0001  significant at 5%
lag 2: F=7.725  p=0.0005  significant at 5%
lag 3: F=4.873  p=0.0023  significant at 5%


### Direction 2: does Sun Pharma's return predict Organon's return?

A standard lagged Granger test here (Sun's return from t-1 predicting Organon's return on t) is valid but conservative - it ignores the fresher, same-calendar-date Sun Pharma close, which by the trading-hours argument above is *also* legitimate prior information for Organon's same-day session. Both are reported: the standard lagged version, and a same-day OLS F-test that is not classical Granger causality (it uses a contemporaneous rather than lagged regressor) but is legitimate given the verified non-overlap in trading hours.

In [8]:

gc_sun_to_ogn = grangercausalitytests(merged[["ogn", "sun"]].values, maxlag=3, verbose=False)
print("standard lagged test (Sun t-1..t-3 -> OGN t):")
for lag in [1, 2, 3]:
    f_stat, p_val, _, _ = gc_sun_to_ogn[lag][0]["ssr_ftest"]
    print(f"  lag {lag}: F={f_stat:.3f}  p={p_val:.4f}  {'significant at 5%' if p_val < 0.05 else 'not significant'}")


standard lagged test (Sun t-1..t-3 -> OGN t):
  lag 1: F=0.324  p=0.5691  not significant
  lag 2: F=1.355  p=0.2582  not significant
  lag 3: F=1.585  p=0.1913  not significant


In [9]:

import statsmodels.api as sm

m = merged.copy()
m["ogn_lag1"] = m["ogn"].shift(1)
m = m.dropna()

X_restricted = sm.add_constant(m[["ogn_lag1"]])
X_unrestricted = sm.add_constant(m[["ogn_lag1", "sun"]])
y = m["ogn"]

res_r = sm.OLS(y, X_restricted).fit()
res_u = sm.OLS(y, X_unrestricted).fit()

f_test = res_u.compare_f_test(res_r)
print(f"same-day OLS F-test (OGN_t ~ OGN_t-1 + SUN_t, vs restricted OGN_t ~ OGN_t-1):")
print(f"  F={f_test[0]:.3f}  p={f_test[1]:.4f}  {'significant at 5%' if f_test[1] < 0.05 else 'not significant'}")
print(f"  SUN_t coefficient: {res_u.params['sun']:.4f}  (t={res_u.tvalues['sun']:.2f})")


same-day OLS F-test (OGN_t ~ OGN_t-1 + SUN_t, vs restricted OGN_t ~ OGN_t-1):


  F=0.001  p=0.9757  not significant
  SUN_t coefficient: 0.0020  (t=0.03)


## Summary

In [10]:

summary = pd.Series({
    "garch_pre_persistence": persistence_pre,
    "garch_pre_longrun_annualised_vol": long_run_annualised_vol,
    "arima_ar1_coefficient": arima_fit.params[1],
    "gc_ogn_to_sun_lag1_pvalue": gc_ogn_to_sun[1][0]["ssr_ftest"][1],
    "gc_sun_to_ogn_lag1_pvalue": gc_sun_to_ogn[1][0]["ssr_ftest"][1],
    "sameday_sun_to_ogn_pvalue": f_test[1],
    "sameday_sun_coefficient": res_u.params["sun"],
})
summary.to_csv(cfg.DATA_FINAL / "timeseries_summary.csv", header=["value"])
summary.round(4)


garch_pre_persistence               0.9224
garch_pre_longrun_annualised_vol    0.4500
arima_ar1_coefficient               0.9582
gc_ogn_to_sun_lag1_pvalue           0.0001
gc_sun_to_ogn_lag1_pvalue           0.5691
sameday_sun_to_ogn_pvalue           0.9757
sameday_sun_coefficient             0.0020
dtype: float64